### Bibliotecas

In [18]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow

# Aumento de dados
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

### Carregando Dados

In [19]:
DATA_DIR = Path("../Datasets")
OUTPUT_DIR = Path("../Datasets/output")
OUTPUT_DIR.mkdir(exist_ok=True)

# --- Hard Failure ---
df_hard = pd.read_csv(DATA_DIR / "HardFailure_dataset.csv")
df_hard["source"] = "hard_failure"
print(f"HardFailure:  {df_hard.shape}")

# --- Soft Failure ---
df_soft = pd.read_csv(DATA_DIR / "SoftFailure_dataset.csv")
df_soft["source"] = "soft_failure"
print(f"SoftFailure:  {df_soft.shape}")

# --- Lightpath QoT ---
df_lp = pd.read_csv(
    DATA_DIR / "Lightpath_756_label_4_QoT_dataset_train_900.txt",
    sep=r"\s+",
    skiprows=2,       # pula linha de descrição e cabeçalho entre aspas
    header=None,
    names=["Timestamp", "LP_length_km", "Laser_current_mA", "LP_power_dBm", "OSNR", "BER_dB", "Failure_type"]
)
df_lp["source"] = "lightpath"
print(f"Lightpath:    {df_lp.shape}")

df_hard.head(3)

HardFailure:  (65733, 9)
SoftFailure:  (53697, 9)
Lightpath:    (2721600, 8)


,Timestamp,Type,ID,BER,OSNR,InputPower,OutputPower,Failure,source
0,1623394635,Infrastructure,Ampli1,NaN,NaN,-23.0,0.7,NaN,hard_failure
1,1623394634,Devices,SPO1/18/11,2.280000e-08,38.5,NaN,NaN,NaN,hard_failure
2,1623394635,Devices,SPO2/18/11,8.730000e-07,23.5,NaN,NaN,NaN,hard_failure


### Preparação dos dados

#### Junção DFS

In [20]:
# Normaliza colunas do Hard/Soft para o esquema unificado
# Renomeia Failure → Failure_type com mapeamento binário (0=Normal, 1=Falha)
for df in [df_hard, df_soft]:
    df.rename(columns={"Failure": "Failure_type"}, inplace=True)

# Lightpath: BER está em dB (escala logarítmica). Hard/Soft: BER é razão linear.
# Convertemos o BER do Lightpath para razão linear para homogeneizar:
#   BER_ratio = 10 ^ (BER_dB / 10)
# Nota: valores como -267 dB resultam em ~0, representando praticamente zero erros.
df_lp["BER"] = 10 ** (df_lp["BER_dB"] / 10)
df_lp.drop(columns=["BER_dB"], inplace=True)

# Concatena os três datasets alinhando colunas (NaN onde não houver valor)
df = pd.concat([df_hard, df_soft, df_lp], ignore_index=True, sort=False)
print(f"Dataset unificado: {df.shape}")
print(f"\nColunas: {df.columns.tolist()}")
print(f"\nDistribuição por source:\n{df['source'].value_counts()}")
df.head()

Dataset unificado: (2841030, 12)

Colunas: ['Timestamp', 'Type', 'ID', 'BER', 'OSNR', 'InputPower', 'OutputPower', 'Failure_type', 'source', 'LP_length_km', 'Laser_current_mA', 'LP_power_dBm']

Distribuição por source:
source
lightpath       2721600
hard_failure      65733
soft_failure      53697
Name: count, dtype: int64


,Timestamp,Type,ID,BER,OSNR,InputPower,OutputPower,Failure_type,source,LP_length_km,Laser_current_mA,LP_power_dBm
0,1623394635,Infrastructure,Ampli1,NaN,NaN,-23.0,0.7,NaN,hard_failure,NaN,NaN,NaN
1,1623394634,Devices,SPO1/18/11,2.280000e-08,38.5,NaN,NaN,NaN,hard_failure,NaN,NaN,NaN
2,1623394635,Devices,SPO2/18/11,8.730000e-07,23.5,NaN,NaN,NaN,hard_failure,NaN,NaN,NaN
3,1623394635,Infrastructure,Ampli2,NaN,NaN,-15.5,0.4,NaN,hard_failure,NaN,NaN,NaN
4,1623394635,Infrastructure,Ampli4,NaN,NaN,-22.9,0.7,NaN,hard_failure,NaN,NaN,NaN


#### Conversão para datetime

In [21]:
# Hard/Soft: Timestamp é Unix epoch (segundos) → datetime
mask_hs = df["source"].isin(["hard_failure", "soft_failure"])
df.loc[mask_hs, "Timestamp"] = pd.to_datetime(df.loc[mask_hs, "Timestamp"], unit="s")

# Lightpath: Timestamp é sequencial (1, 2, 3...) → mantém como inteiro,
# mas converte para datetime relativo ao início de hard_failure para comparabilidade
lp_start = pd.to_datetime(df.loc[mask_hs, "Timestamp"].min())
df.loc[~mask_hs, "Timestamp"] = lp_start + pd.to_timedelta(df.loc[~mask_hs, "Timestamp"].astype(int), unit="s")

df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df.sort_values("Timestamp", inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Período: {df['Timestamp'].min()} → {df['Timestamp'].max()}")
df.dtypes

Período: 2021-06-11 06:57:14 → 2021-06-23 22:14:11


Timestamp           datetime64[ns]
Type                        object
ID                          object
BER                        float64
OSNR                       float64
InputPower                 float64
OutputPower                float64
Failure_type               float64
source                      object
LP_length_km               float64
Laser_current_mA           float64
LP_power_dBm               float64
dtype: object

#### Replice Failure - NaN = 0


In [22]:
# Ausência de rótulo → operação normal
df["Failure_type"] = df["Failure_type"].fillna(0).astype(int)

print("Distribuição de Failure_type:")
print(df["Failure_type"].value_counts().sort_index())
print(f"\n0 = Normal | 1 = Falha (Hard/Soft) | 1=ECL / 2=EDFA / 3=NLI (Lightpath)")

Distribuição de Failure_type:
Failure_type
0    792219
1    688011
2    680400
3    680400
Name: count, dtype: int64

0 = Normal | 1 = Falha (Hard/Soft) | 1=ECL / 2=EDFA / 3=NLI (Lightpath)


### Interpolação - Tratamento de NaN nas Features Numéricas

In [23]:
numeric_cols = ["BER", "OSNR", "InputPower", "OutputPower", "LP_length_km", "Laser_current_mA", "LP_power_dBm"]

# Interpolação linear por source (respeita a continuidade de cada série)
df[numeric_cols] = (
    df.groupby("source")[numeric_cols]
    .transform(lambda s: s.interpolate(method="linear", limit_direction="both"))
)

missing_after = df[numeric_cols].isna().sum()
print("NaN restantes após interpolação:")
print(missing_after[missing_after > 0] if missing_after.any() else "Nenhum NaN restante")
df.info()

NaN restantes após interpolação:
InputPower          2721600
OutputPower         2721600
LP_length_km         119430
Laser_current_mA     119430
LP_power_dBm         119430
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2841030 entries, 0 to 2841029
Data columns (total 12 columns):
 #   Column            Dtype         
---  ------            -----         
 0   Timestamp         datetime64[ns]
 1   Type              object        
 2   ID                object        
 3   BER               float64       
 4   OSNR              float64       
 5   InputPower        float64       
 6   OutputPower       float64       
 7   Failure_type      int64         
 8   source            object        
 9   LP_length_km      float64       
 10  Laser_current_mA  float64       
 11  LP_power_dBm      float64       
dtypes: datetime64[ns](1), float64(7), int64(1), object(3)
memory usage: 260.1+ MB


### Aumento de Dados

Estratégia em duas etapas:
1. **SMOTE** — gera amostras sintéticas das classes minoritárias (falhas) para equilibrar a base
2. **Ruído Gaussiano** — adiciona pequenas perturbações às amostras originais para aumentar a diversidade

In [24]:
smote_features = [c for c in numeric_cols if df[c].notna().all()]
print(f"Features usadas no SMOTE: {smote_features}")

X = df[smote_features].values
y = df["Failure_type"].values

print(f"\nDistribuição antes do SMOTE:")
unique, counts = np.unique(y, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  Classe {cls}: {cnt:,} ({cnt/len(y)*100:.1f}%)")

Features usadas no SMOTE: ['BER', 'OSNR']

Distribuição antes do SMOTE:
  Classe 0: 792,219 (27.9%)
  Classe 1: 688,011 (24.2%)
  Classe 2: 680,400 (23.9%)
  Classe 3: 680,400 (23.9%)


In [25]:
# SMOTE — k_neighbors=5 padrão; aumenta classes minoritárias até ~20% da classe majoritária
# (estratégia conservadora: não força 50/50 para não distorcer a distribuição real)
class_counts = dict(zip(*np.unique(y, return_counts=True)))
majority_count = max(class_counts.values())
sampling_strategy = {
    cls: max(cnt, int(majority_count * 0.20))
    for cls, cnt in class_counts.items()
    if cls != 0  # não reamostra a classe normal
}

smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42, k_neighbors=5)
X_sm, y_sm = smote.fit_resample(X, y)

df_smote = pd.DataFrame(X_sm, columns=smote_features)
df_smote["Failure_type"] = y_sm.astype(int)
df_smote["source"] = "synthetic_smote"

print(f"SMOTE gerou {len(df_smote) - len(df):,} amostras sintéticas")
print(f"\nDistribuição após SMOTE:")
unique_sm, counts_sm = np.unique(y_sm, return_counts=True)
for cls, cnt in zip(unique_sm, counts_sm):
    print(f"  Classe {cls}: {cnt:,} ({cnt/len(y_sm)*100:.1f}%)")

SMOTE gerou 0 amostras sintéticas

Distribuição após SMOTE:
  Classe 0: 792,219 (27.9%)
  Classe 1: 688,011 (24.2%)
  Classe 2: 680,400 (23.9%)
  Classe 3: 680,400 (23.9%)


In [26]:
# Ruído Gaussiano — perturbação de 1% do desvio padrão de cada feature
# Aplicado sobre os dados originais (não sobre os sintéticos do SMOTE)
rng = np.random.default_rng(42)
noise_scale = 0.01

df_orig_numeric = df[smote_features].copy()
noise = rng.normal(loc=0, scale=noise_scale, size=df_orig_numeric.shape)
noise *= df_orig_numeric.std().values  # escala pelo std de cada coluna

df_noise = df_orig_numeric + noise
df_noise["Failure_type"] = df["Failure_type"].values
df_noise["source"] = "synthetic_gaussian"

print(f"Amostras com ruído gaussiano: {len(df_noise):,}")

Amostras com ruído gaussiano: 2,841,030


In [27]:
# Consolida: original + SMOTE sintético + gaussiano
# (apenas as linhas novas do SMOTE, não duplica os originais)
n_original = len(df)
df_smote_only = df_smote.iloc[n_original:].copy()  # apenas as linhas sintéticas

df_final = pd.concat([df, df_smote_only, df_noise], ignore_index=True, sort=False)

print(f"Base final: {df_final.shape[0]:,} linhas × {df_final.shape[1]} colunas")
print(f"\nDistribuição de Failure_type:")
print(df_final["Failure_type"].value_counts().sort_index())
print(f"\nDistribuição por source:")
print(df_final["source"].value_counts())

Base final: 5,682,060 linhas × 12 colunas

Distribuição de Failure_type:
Failure_type
0    1584438
1    1376022
2    1360800
3    1360800
Name: count, dtype: int64

Distribuição por source:
source
synthetic_gaussian    2841030
lightpath             2721600
hard_failure            65733
soft_failure            53697
Name: count, dtype: int64


### Tratamento de Nulos Residuais

Os nulos restantes são **estruturais**: colunas exclusivas de um dataset não existem nas outras fontes (ex: `LP_length_km` não existe para Hard/Soft; `InputPower` não existe para Lightpath).

In [28]:
null_summary = df_final.isna().sum()
null_summary = null_summary[null_summary > 0].sort_values(ascending=False)
print("Colunas com nulos:")
print(null_summary.to_string())
print(f"\nTotal de linhas: {len(df_final):,}")

Colunas com nulos:
Type                5562630
ID                  5562630
InputPower          5562630
OutputPower         5562630
Laser_current_mA    2960460
LP_length_km        2960460
LP_power_dBm        2960460
Timestamp           2841030

Total de linhas: 5,682,060


In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor

# Colunas categóricas: não há modelo numérico para isso → preenche com "N/A"
cat_cols = ["Type", "ID"]
for col in cat_cols:
    if col in df_final.columns:
        df_final[col] = df_final[col].fillna("N/A")

# Colunas numéricas com nulos estruturais
impute_cols = ["BER", "OSNR", "InputPower", "OutputPower",
               "LP_length_km", "Laser_current_mA", "LP_power_dBm"]
impute_cols = [c for c in impute_cols if c in df_final.columns]

# IterativeImputer com ExtraTreesRegressor:
# para cada coluna com nulo, treina uma árvore usando as demais como features.
# max_iter=10 é suficiente para convergência na maioria dos casos.
imputer = IterativeImputer(
    estimator=ExtraTreesRegressor(n_estimators=10, random_state=42),
    max_iter=10,
    random_state=42,
    verbose=1
)

df_final[impute_cols] = imputer.fit_transform(df_final[impute_cols])

remaining = df_final[impute_cols].isna().sum().sum()
print(f"\nNulos restantes nas colunas numéricas: {remaining}")
df_final[impute_cols].describe()

[IterativeImputer] Completing matrix with shape (5682060, 7)
[IterativeImputer] Change: 3078.941899554081, scaled tolerance: 5.051 
[IterativeImputer] Change: 1435.495096280775, scaled tolerance: 5.051 


### Exportação da Base

In [ ]:
csv_path = OUTPUT_DIR / "dataset_unificado.csv"
parquet_path = OUTPUT_DIR / "dataset_unificado.parquet"

df_final.to_csv(csv_path, index=False)
csv_mb = csv_path.stat().st_size / 1e6
print(f"CSV    → {csv_path}  ({csv_mb:.1f} MB)")

try:
    df_final.to_parquet(parquet_path, index=False, engine="fastparquet")
except Exception:
    try:
        df_final.to_parquet(parquet_path, index=False, engine="pyarrow")
    except Exception as e:
        print(f"Parquet não salvo ({e.__class__.__name__}): reinicie o kernel e tente novamente")
        parquet_path = None

if parquet_path and parquet_path.exists():
    pq_mb = parquet_path.stat().st_size / 1e6
    print(f"Parquet→ {parquet_path}  ({pq_mb:.1f} MB)")

print(f"\nResumo final:")
print(df_final.describe())